# Chapter 15 Companion — Systemic Risk Tunneling: Integrated Synthesis

## Introduction: the five central ideas

1. **Systemic tunneling integrates geometry, dynamics, barriers, pathways, measurement, and governance.**
2. **Microprudential safety does not guarantee macroprudential stability.**
3. **Network structure and synchronized behavior can create collective hidden pathways.**
4. **Empirical implementation combines institution-level states with exposure networks and scenario dynamics.**
5. **The final objective is intervention leverage: where a small action most reduces systemic permeability.**

**Empirical objective.** Demonstrate why individually safe institutions can still generate systemic fragility. The notebook uses deterministic synthetic data so that every transformation, estimate, and diagnostic can be inspected. It contains exactly ten executable cells; every cell is preceded by an explanation.

## Cell 1 — Setup and empirical contract

This cell fixes the random seed, imports lightweight scientific libraries, defines the chapter-specific axes, and records the empirical contract. Reproducibility and explicit assumptions begin before any data are generated.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, json, math
np.random.seed(758 + 15)
TITLE='Systemic Risk Tunneling: Integrated Synthesis'; XLABEL='local robustness'; YLABEL='system connectivity'
N=720; TRAIN_END=540; BARRIER_HEIGHT=1.44; BARRIER_WIDTH=0.41
plt.style.use("seaborn-v0_8-whitegrid")
print({"chapter":15,"title":TITLE,"seed":758+15,"observations":N,"train_end":TRAIN_END})

## Cell 2 — Generate time-ordered risk observations

This cell creates a synthetic sequence with gradual drift, correlated shocks, regime stress, and a binary adverse transition. The construction supplies a known but nontrivial environment in which the chapter's mechanisms can be inspected.

In [ ]:
t=np.arange(N); regime=(t>360).astype(float)
eps=np.random.multivariate_normal([0,0],[[0.035,0.018],[0.018,0.045]],N)
x=np.zeros(N); y=np.zeros(N); x[0],y[0]=-1.25,-0.85
for i in range(1,N):
    x[i]=0.985*x[i-1]+0.004+0.018*regime[i]+eps[i,0]
    y[i]=0.980*y[i-1]+0.003+0.015*regime[i]+0.14*np.sin(i/47)+eps[i,1]
distance=1.15-(0.72*x+0.55*y)
weak_path=np.exp(-((y-0.25)/BARRIER_WIDTH)**2)
latent=-1.1+0.85*x+0.70*y+1.15*weak_path+0.35*regime
p_true=1/(1+np.exp(-latent)); event=(np.random.rand(N)<p_true).astype(int)
print({"event_rate":round(event.mean(),3),"train_rate":round(event[:TRAIN_END].mean(),3),"holdout_rate":round(event[TRAIN_END:].mean(),3)})

## Cell 3 — Visualize the state manifold

This cell maps the observations into the chapter's two-dimensional state space. Time and adverse outcomes are shown explicitly so that position, direction, clusters, and nonlinear boundaries become visible.

In [ ]:
fig,ax=plt.subplots(figsize=(9,6))
sc=ax.scatter(x,y,c=t,cmap="viridis",s=18,alpha=.7); ax.scatter(x[event==1],y[event==1],facecolors="none",edgecolors="crimson",s=45,label="adverse transition")
ax.set(xlabel=XLABEL,ylabel=YLABEL,title=TITLE+": time-ordered state manifold"); ax.legend(); plt.colorbar(sc,ax=ax,label="time"); plt.show()

## Cell 4 — Estimate the risk landscape

This cell constructs a smooth potential surface and its contour geometry. It demonstrates how equal-risk contours, stable basins, saddles, and adverse regions carry more information than one probability number.

In [ ]:
gx,gy=np.meshgrid(np.linspace(x.min()-.4,x.max()+.4,180),np.linspace(y.min()-.4,y.max()+.4,180))
potential=0.32*(gx+1.0)**2+0.28*(gy+.7)**2+1.35/(1+np.exp(-3*(.72*gx+.55*gy-1.15)))
fig,ax=plt.subplots(figsize=(9,6)); c=ax.contourf(gx,gy,potential,30,cmap="terrain"); ax.contour(gx,gy,potential,12,colors="k",alpha=.25,linewidths=.5)
ax.set(xlabel=XLABEL,ylabel=YLABEL,title="Estimated risk topography: basins, slopes and adverse ridge"); plt.colorbar(c,ax=ax,label="potential risk"); plt.show()

## Cell 5 — Represent the protective barrier

This cell measures barrier height, width, local weakness, and distance. It also computes a tunneling-style permeability index, making the chapter's protective structure numerically inspectable.

In [ ]:
energy=np.maximum(0.03,0.55+0.22*x+0.18*y)
local_height=BARRIER_HEIGHT*(1-.52*weak_path)
action=np.sqrt(np.maximum(local_height-energy,0))*BARRIER_WIDTH
permeability=np.exp(-2*action)
fig,ax=plt.subplots(1,2,figsize=(12,4)); ax[0].plot(t,local_height,label="local barrier"); ax[0].plot(t,energy,label="system energy"); ax[0].legend(); ax[0].set_title("Barrier versus pressure")
ax[1].plot(t,permeability,color="purple"); ax[1].set_title("Tunneling-style permeability"); ax[1].set_ylim(0,1.03); plt.show()
print({"median_permeability":round(float(np.median(permeability)),3),"p95_permeability":round(float(np.quantile(permeability,.95)),3)})

## Cell 6 — Simulate classical and hidden-path transitions

This cell advances many institutions through the landscape. It contrasts ordinary threshold crossing with transitions activated through a narrow, state-dependent pathway.

In [ ]:
M=5000; rng=np.random.default_rng(1000+int(BARRIER_HEIGHT*100)); z=rng.normal(size=(M,2))*0.55+[-.7,-.55]
classical=(.72*z[:,0]+.55*z[:,1]>1.15)
path_activation=np.exp(-((z[:,1]-.25)/BARRIER_WIDTH)**2)*np.exp(-2*np.sqrt(np.maximum(BARRIER_HEIGHT-(.55+.22*z[:,0]+.18*z[:,1]),0))*BARRIER_WIDTH)
hidden=(rng.random(M)<.18*path_activation)&(~classical); total=classical|hidden
print({"classical_rate":round(classical.mean(),4),"hidden_path_rate":round(hidden.mean(),4),"total_transition_rate":round(total.mean(),4),"hidden_share":round(hidden.sum()/max(total.sum(),1),3)})

## Cell 7 — Estimate parameters on a chronological training window

This cell performs empirical calibration without using future observations. The fitted conventional score and geometry-augmented score are frozen before the holdout is examined.

In [ ]:
def sigmoid(z): return 1/(1+np.exp(-np.clip(z,-30,30)))
def fit_logit(X,y,steps=2200,lr=.12):
    X=np.column_stack([np.ones(len(X)),X]); b=np.zeros(X.shape[1])
    for _ in range(steps): b-=lr*(X.T@(sigmoid(X@b)-y)/len(y)+1e-3*b)
    return b
X0=np.column_stack([x,y,regime]); Xg=np.column_stack([x,y,regime,distance,weak_path,permeability])
b0=fit_logit(X0[:TRAIN_END],event[:TRAIN_END]); bg=fit_logit(Xg[:TRAIN_END],event[:TRAIN_END])
print("conventional coefficients",np.round(b0,3)); print("geometric coefficients",np.round(bg,3))

## Cell 8 — Stress and sensitivity analysis

This cell varies barrier strength and pathway width across a grid. The heatmap reveals nonlinear regions in which small structural changes produce large changes in permeability.

In [ ]:
hs=np.linspace(.7,2.2,45); ws=np.linspace(.12,.75,45); H,W=np.meshgrid(hs,ws)
E=.78; sensitivity=np.exp(-2*np.sqrt(np.maximum(H-E,0))*W)
fig,ax=plt.subplots(figsize=(9,6)); im=ax.contourf(H,W,sensitivity,25,cmap="magma"); ax.scatter([BARRIER_HEIGHT],[BARRIER_WIDTH],c="cyan",edgecolor="black",s=90,label="chapter baseline")
ax.set(xlabel="barrier height",ylabel="pathway width",title="Permeability sensitivity surface"); ax.legend(); plt.colorbar(im,ax=ax,label="permeability"); plt.show()

## Cell 9 — Locked holdout comparison

This cell evaluates both models on the final time-ordered observations using Brier score and event-rate calibration. The comparison asks whether the geometric variables add empirical information beyond a conventional scalar benchmark.

In [ ]:
idx=np.arange(TRAIN_END,N); p0=sigmoid(np.column_stack([np.ones(len(idx)),X0[idx]])@b0); pg=sigmoid(np.column_stack([np.ones(len(idx)),Xg[idx]])@bg)
brier0=np.mean((p0-event[idx])**2); brierg=np.mean((pg-event[idx])**2)
bins=np.linspace(0,1,6); fig,ax=plt.subplots(figsize=(7,5))
for pred,label in [(p0,"conventional"),(pg,"geometry augmented")]:
    centers=[]; observed=[]
    for a,b in zip(bins[:-1],bins[1:]):
        m=(pred>=a)&(pred<(b+1e-9));
        if m.sum(): centers.append(pred[m].mean()); observed.append(event[idx][m].mean())
    ax.plot(centers,observed,"o-",label=label)
ax.plot([0,1],[0,1],"k--"); ax.set(xlabel="predicted rate",ylabel="observed rate",title="Locked chronological holdout calibration"); ax.legend(); plt.show()
print({"brier_conventional":round(float(brier0),4),"brier_geometry":round(float(brierg),4),"improvement_pct":round(float(100*(brier0-brierg)/brier0),2)})

## Cell 10 — Audit manifest and limitations

This final cell records parameters, results, assumptions, and limitations in a machine-readable manifest. It keeps source construction, estimates, outcomes, and interpretation separate.

In [ ]:
manifest={"chapter":15,"title":TITLE,"seed":758+15,"n_observations":N,"train_end":TRAIN_END,"holdout_n":N-TRAIN_END,"barrier_height":BARRIER_HEIGHT,"barrier_width":BARRIER_WIDTH,"brier_conventional":float(brier0),"brier_geometry":float(brierg),"limitations":["synthetic data","two-dimensional pedagogical projection","tunneling is a formal risk analogy","parameters require domain validation","human review required before decisions"]}
print(json.dumps(manifest,indent=2))
assert len(idx)==180 and np.isfinite([brier0,brierg]).all()
print("AUDIT CHECKS PASSED")

## Conclusion

This companion notebook converted Chapter 15, **Systemic Risk Tunneling: Integrated Synthesis**, into a reproducible empirical experiment. The central lesson is that a risk system cannot be understood only through a terminal event probability. The notebook represented the institution as a location in a state space, estimated its movement, constructed a protective barrier, identified possible pathways, and measured the probability of reaching an adverse region under both ordinary and tunneling-sensitive descriptions. This sequence makes the chapter's abstract objects observable: coordinates become measurable indicators; geometry becomes a covariance-aware distance; barrier strength becomes an estimated function; and permeability becomes a scenario-dependent quantity.

The experiment also demonstrated why implementation discipline matters. All observations were synthetic and generated with a fixed seed, so the logic is transparent and exactly reproducible. The chronological split prevents future observations from informing the past. The conventional benchmark and the geometry-augmented score were evaluated on the same locked holdout, allowing incremental value to be assessed rather than assumed. Sensitivity analysis showed which parameters dominate the result, while the audit manifest records assumptions, sample sizes, split location, and model limitations.

These outputs should not be interpreted as evidence that financial systems literally obey quantum mechanics. Tunneling is used here as a formal analogy for discontinuous or hidden-path transitions across apparently protective structures. Real implementation would require validated domain data, defensible proxies, uncertainty quantification, stability tests, and expert review. Measurement error can distort the state manifold; omitted channels can create false confidence; and a fitted barrier can change when institutions adapt.

Within the book's cumulative journey, Chapter 15 therefore adds a computational layer to the geometric language of risk. It shows not only what the concepts mean, but how an analyst can estimate, visualize, challenge, and compare them. The result is a practical bridge from conceptual geometry to empirical risk management: a model that remains inspectable, testable, and explicit about where its conclusions stop.